# Customer Segmentation using K-Means Clustering

**Task:** Group customers of a retail store based on their purchase history using K-Means clustering.

This notebook covers:
1. Load / generate data
2. Explore the data
3. Preprocess (scale features)
4. Find the optimal number of clusters (Elbow Method)
5. Train the K-Means model
6. Visualize and interpret the customer segments


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

np.random.seed(42)


## 2. Load the Dataset

**If you have a real dataset** (e.g. Kaggle's 'Mall Customers' dataset), replace the next cell with:
```python
df = pd.read_csv('Mall_Customers.csv')
df = df[['Annual Income (k$)', 'Spending Score (1-100)']]
df.columns = ['annual_income', 'spending_score']
```
For now, we'll generate a **synthetic retail purchase-history dataset** so the notebook runs end-to-end without any external file. It includes features that reflect actual purchase behavior:
- `annual_spend`: total amount spent per year
- `purchase_frequency`: number of purchases per year
- `avg_basket_value`: average amount spent per purchase


In [ ]:
# --- Synthetic dataset generator (replace with pd.read_csv(...) for real data) ---
n_customers = 400

# Create a few natural customer segments to make clustering meaningful
segment_sizes = [100, 120, 100, 80]
segments = []

# Segment 1: low spend, infrequent buyers
segments.append(pd.DataFrame({
    'annual_spend': np.random.normal(300, 80, segment_sizes[0]),
    'purchase_frequency': np.random.normal(4, 1.5, segment_sizes[0]),
    'avg_basket_value': np.random.normal(70, 15, segment_sizes[0])
}))

# Segment 2: high spend, frequent buyers (loyal big spenders)
segments.append(pd.DataFrame({
    'annual_spend': np.random.normal(2500, 400, segment_sizes[1]),
    'purchase_frequency': np.random.normal(30, 5, segment_sizes[1]),
    'avg_basket_value': np.random.normal(85, 20, segment_sizes[1])
}))

# Segment 3: moderate spend, occasional big-basket buyers
segments.append(pd.DataFrame({
    'annual_spend': np.random.normal(1200, 250, segment_sizes[2]),
    'purchase_frequency': np.random.normal(8, 2, segment_sizes[2]),
    'avg_basket_value': np.random.normal(150, 25, segment_sizes[2])
}))

# Segment 4: frequent small purchases (bargain shoppers)
segments.append(pd.DataFrame({
    'annual_spend': np.random.normal(900, 150, segment_sizes[3]),
    'purchase_frequency': np.random.normal(25, 4, segment_sizes[3]),
    'avg_basket_value': np.random.normal(36, 8, segment_sizes[3])
}))

df = pd.concat(segments, ignore_index=True)
df = df.clip(lower=0)  # no negative values
df['customer_id'] = range(1, len(df) + 1)
df = df[['customer_id', 'annual_spend', 'purchase_frequency', 'avg_basket_value']]

df.head()


## 3. Explore the Data

In [ ]:
df.describe()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['annual_spend'], bins=30)
axes[0].set_title('Annual Spend Distribution')

axes[1].hist(df['purchase_frequency'], bins=30)
axes[1].set_title('Purchase Frequency Distribution')

axes[2].hist(df['avg_basket_value'], bins=30)
axes[2].set_title('Avg Basket Value Distribution')

plt.tight_layout()
plt.show()


## 4. Preprocess: Scale the Features

K-Means uses distance between points to form clusters, so features need to be on a similar scale — otherwise a feature like `annual_spend` (hundreds/thousands) would dominate a feature like `purchase_frequency` (single digits/tens).

In [ ]:
features = ['annual_spend', 'purchase_frequency', 'avg_basket_value']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## 5. Find the Optimal Number of Clusters

We use the **Elbow Method**: plot inertia (within-cluster sum of squares) against different values of k, and look for the point where the improvement starts to level off — that 'elbow' is a good choice for k.

In [ ]:
inertias = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertias, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Also check silhouette score for a more objective confirmation of k
sil_scores = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), sil_scores, marker='o', color='green')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for Different k')
plt.grid(True, alpha=0.3)
plt.show()

best_k = range(2, 11)[np.argmax(sil_scores)]
print(f'Best k based on silhouette score: {best_k}')


## 6. Train the Final K-Means Model

Based on the elbow plot and silhouette score, we'll go with **k = 4** clusters (adjust this based on what your own elbow/silhouette plots show).

In [ ]:
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

df.head()


## 7. Visualize the Clusters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

scatter1 = axes[0].scatter(df['annual_spend'], df['purchase_frequency'], c=df['cluster'], cmap='viridis', alpha=0.6)
axes[0].set_xlabel('Annual Spend')
axes[0].set_ylabel('Purchase Frequency')
axes[0].set_title('Clusters: Spend vs Frequency')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

scatter2 = axes[1].scatter(df['annual_spend'], df['avg_basket_value'], c=df['cluster'], cmap='viridis', alpha=0.6)
axes[1].set_xlabel('Annual Spend')
axes[1].set_ylabel('Avg Basket Value')
axes[1].set_title('Clusters: Spend vs Basket Value')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.show()


## 8. Interpret the Segments

Let's look at the average behavior per cluster to give each one a business-friendly label.

In [ ]:
cluster_summary = df.groupby('cluster')[features].mean().round(2)
cluster_summary['count'] = df['cluster'].value_counts().sort_index()
cluster_summary


**Typical segment interpretation** (yours may vary depending on the random data / real dataset used):
- **High spend + high frequency** → Loyal / VIP customers
- **High spend + low frequency but high basket value** → Occasional big-ticket buyers
- **Low spend + low frequency** → Low-engagement / at-risk customers
- **Moderate spend + high frequency but low basket value** → Frequent bargain shoppers

This kind of segmentation helps a retail store target each group differently — e.g. loyalty rewards for VIPs, win-back campaigns for at-risk customers, and bundle deals for bargain shoppers.

## Summary

- We used **K-Means clustering** to segment retail customers based on purchase history: annual spend, purchase frequency, and average basket value.
- We scaled features first (important for distance-based algorithms like K-Means).
- We used the **Elbow Method** and **Silhouette Score** to choose a sensible number of clusters.
- We visualized and interpreted the resulting customer segments in business terms.

**Next steps to make this more robust for a real submission:**
- Swap in a real dataset (e.g. Kaggle's Mall Customers dataset, or actual store transaction data)
- Try more features (recency of last purchase, product category diversity, etc.)
- Try other clustering algorithms (DBSCAN, Hierarchical Clustering) and compare
- Use PCA to visualize clusters in 2D if you have more than 2-3 features
